<a href="https://colab.research.google.com/github/chaunijs/onlineshoppingprice/blob/main/notebook_ipynb%20/fix_new_bigc.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## BigC Online Product Scraper

This notebook contains a web scraper for BigC products. It is designed to fetch product information from specified URLs, process the data, and save it to `.xlsx` and `.csv` files.

In [43]:
import subprocess
import sys
from IPython.display import display, HTML

# 1. Setup the 2-line scrolling box UI
display(HTML("""
    <style>
        #scroll_output {
            height: 50px; /* Approximately 2 lines */
            overflow-y: scroll;
            background-color: #1e1e1e;
            color: #00ff00;
            padding: 10px;
            font-family: monospace;
            font-size: 14px;
            border: 1px solid #444;
            display: flex;
            flex-direction: column;
        }
    </style>
    <div id="scroll_output">Starting installation...</div>
"""))

def run_and_scroll(commands):
    """Runs list of commands and streams output to the scroll_output div"""
    for cmd in commands:
        process = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in process.stdout:
            # Escape single quotes for JS and update the div
            escaped_line = line.replace("'", "\\'").strip()
            display(HTML(f"""<script>
                var obj = document.getElementById('scroll_output');
                obj.innerHTML += '<div>' + '{escaped_line}' + '</div>';
                obj.scrollTop = obj.scrollHeight;
            </script>"""), display_id='scroll_update')
        process.wait()

# 2. Updated list of commands (Consolidated and Optimized)
commands_to_run = [
    # 1. Download and install the official Google Chrome stable version
    # "wget -q -O - https://dl-ssl.google.com/linux/linux_signing_key.pub | apt-key add -",
    # "sh -c 'echo \"deb [arch=amd64] http://dl.google.com/linux/chrome/deb/ stable main\" >> /etc/apt/sources.list.d/google-chrome.list'",
    "apt-get -y update",
    "apt-get install -y google-chrome-stable",

    # 2. Install all Python dependencies in one step (quiet mode)
    "pip install selenium beautifulsoup4 pandas polars playwright chromedriver-autoinstaller xlsxwriter fastexcel curl_cffi scrapling patchright msgspec browserforge nest_asyncio scrapling -q",

    # 3. Install Playwright and Patchright browsers and their OS dependencies
    "playwright install chromium",
    "playwright install-deps",
    "patchright install chromium",
    "patchright install-deps"
]

run_and_scroll(commands_to_run)
print("\n✅ All installations finished.")


✅ All installations finished.


In [45]:
from playwright.async_api import async_playwright
from bs4 import BeautifulSoup
import polars as pl
import asyncio
import xlsxwriter
import datetime
from datetime import date
import IPython.display as display
import datetime
today_date = datetime.datetime.now().strftime("%Y-%m-%d")
print("Today is",today_date)

Today is 2026-08-24


In [61]:
import time
import pandas as pd
from scrapling.fetchers import StealthyFetcher

# ----------------------------------------------------------------------
# CONFIG
# ----------------------------------------------------------------------
# Set limit=90 to load 90 products per request
URLS = [
    "https://www.bigc.co.th/en/category/laundry?page={page}&limit=90",
]

DELAY_SECONDS = 2
OUTPUT_XLSX = "bigc_products.xlsx"
OUTPUT_CSV = "bigc_products.csv"
PRODUCT_CARD_SELECTOR = "main ul > li"

# ----------------------------------------------------------------------
# HELPERS
# ----------------------------------------------------------------------
def clean_price(value):
    """Strip currency symbol, commas and whitespace; return clean string."""
    if not value:
        return None
    return value.replace("฿", "").replace(",", "").strip()

def extract_product(item):
    """Extract one product's fields from a single card element."""
    # Product Name
    name = item.css('p[class*="line-clamp-2"]::text').get()

    # 1. Promotion Price
    promotion_price = item.css('p[class*="text-red-500"] span[class*="text-xl"]::text').get()
    if not promotion_price:
        red_spans = item.css('p[class*="text-red-500"] span::text').getall()
        promotion_price = "".join([s for s in red_spans if '฿' not in s]).strip() or None

    # 2. Original Price
    original_price = item.css('div[class*="line-through"]::text').get()

    # 3. Badges / Conditions (Excludes Nextday / delivery tags)
    condition_spans = item.css('a div span::text').getall()
    conditions = [
        s.strip() for s in condition_spans
        if s.strip()
        and s.strip() != '฿'
        and not s.strip().startswith('-')
        and s.strip().lower() not in ["nextday", "next day"]
    ]
    condition = " | ".join(list(dict.fromkeys(conditions))) if conditions else None

    return {
        "product_name": name.strip() if name else None,
        "promotion_price": clean_price(promotion_price),
        "original_price": clean_price(original_price),
        "condition": condition,
    }

# ----------------------------------------------------------------------
# SCRAPER
# ----------------------------------------------------------------------
def scrape():
    all_data = []
    for base_url in URLS:
        page = 1
        while True:
            url = base_url.format(page=page) if "{page}" in base_url else base_url
            print(f"[*] Fetching: {url}")

            page_result = StealthyFetcher.fetch(url, headless=True, network_idle=True)
            containers = page_result.css(PRODUCT_CARD_SELECTOR)

            if not containers:
                print(f"    -> No product cards found on page {page}. Category complete.")
                break

            print(f"    -> Found {len(containers)} products")

            for item in containers:
                data = extract_product(item)
                if data["product_name"]:
                    all_data.append(data)

            time.sleep(DELAY_SECONDS)
            if "{page}" not in base_url:
                break
            page += 1

    return all_data

The following cell contains the core scraping logic. I've removed the `if __name__ == '__main__':` block so it can be executed directly within Colab.

In [30]:
import concurrent.futures

# Run the scrape function in a background thread
with concurrent.futures.ThreadPoolExecutor() as executor:
    future = executor.submit(scrape)
    rows = future.result()

if not rows:
    print("No data scraped. Check PRODUCT_CARD_SELECTOR and URLs.")
else:
    df_scrape = pl.DataFrame(rows).unique()

    print(f"\n[✓] Scraped {len(df_scrape)} products")
    display(df_scrape.head())

[*] Fetching: https://www.bigc.co.th/en/category/laundry?page=1


[2026-08-24 02:32:15] INFO: Fetched (200) <GET https://www.bigc.co.th/en/category/laundry?page=1> (referer: https://www.google.com/)
INFO:scrapling:Fetched (200) <GET https://www.bigc.co.th/en/category/laundry?page=1> (referer: https://www.google.com/)


    -> Found 60 products
[*] Fetching: https://www.bigc.co.th/en/category/laundry?page=2


[2026-08-24 02:32:24] INFO: Fetched (200) <GET https://www.bigc.co.th/en/category/laundry?page=2> (referer: https://www.google.com/)
INFO:scrapling:Fetched (200) <GET https://www.bigc.co.th/en/category/laundry?page=2> (referer: https://www.google.com/)


    -> Found 60 products
[*] Fetching: https://www.bigc.co.th/en/category/laundry?page=3


...

[2026-08-24 02:39:07] INFO: Fetched (200) <GET https://www.bigc.co.th/en/category/laundry?page=36> (referer: https://www.google.com/)
INFO:scrapling:Fetched (200) <GET https://www.bigc.co.th/en/category/laundry?page=36> (referer: https://www.google.com/)


    -> Found 10 products
[*] Fetching: https://www.bigc.co.th/en/category/laundry?page=37


[2026-08-24 02:39:14] INFO: Fetched (200) <GET https://www.bigc.co.th/en/category/laundry?page=37> (referer: https://www.google.com/)
INFO:scrapling:Fetched (200) <GET https://www.bigc.co.th/en/category/laundry?page=37> (referer: https://www.google.com/)


    -> No product cards found on page 37. Category complete.

[✓] Scraped 954 products


,product_name,promotion_price,original_price,condition
0,"FINELINE Fabric Softener Sunshine Gold 1,300 ml.",None,None,Buy 1 Get 1
1,HYGIENE Expert Care Concentrated Fabric Soften...,None,None,Super Save
2,BREEZE Pure Happy Fabric Detergent 550 ml.,None,None,Super Save
3,FINELINE Plus Liquid Laundry Detergent Sunny G...,None,None,Buy 1 Get 1
4,HYGIENE Fabric Softener Violet Soft Scent 500 ...,None,None,Super Save


In [31]:
df_scrape

,product_name,promotion_price,original_price,condition
0,"FINELINE Fabric Softener Sunshine Gold 1,300 ml.",None,None,Buy 1 Get 1
1,HYGIENE Expert Care Concentrated Fabric Soften...,None,None,Super Save
2,BREEZE Pure Happy Fabric Detergent 550 ml.,None,None,Super Save
3,FINELINE Plus Liquid Laundry Detergent Sunny G...,None,None,Buy 1 Get 1
4,HYGIENE Fabric Softener Violet Soft Scent 500 ...,None,None,Super Save
...,...,...,...,...
949,HYGIENE Expert Care Concentrated Fabric Soften...,None,None,None
950,BIG C HAPPY PRICE Fabric Softener Sense of Ple...,None,None,Super Save
951,BIG C HAPPY PRICE Washing Powder 3.5 kg.,None,None,None
952,COMFORT Floral Fresh Concentrated Fabric Softe...,None,None,None


In [62]:
import concurrent.futures
import polars as pl
from scrapling.fetchers import StealthyFetcher
import time

def scrape_test():
    all_data = []
    for base_url in URLS:
        page = 1
        while page <= 2:
            url = base_url.format(page=page) if "{page}" in base_url else base_url
            print(f"[*] Fetching Test Page: {url}")

            page_result = StealthyFetcher.fetch(url, headless=True, network_idle=True)
            containers = page_result.css(PRODUCT_CARD_SELECTOR)

            if not containers:
                print(f"    -> No product cards found on page {page}.")
                break

            print(f"    -> Found {len(containers)} products")

            for item in containers:
                data = extract_product(item)
                if data["product_name"]:
                    all_data.append(data)

            time.sleep(DELAY_SECONDS)
            if "{page}" not in base_url:
                break
            page += 1

    return all_data

# Run test in thread
with concurrent.futures.ThreadPoolExecutor() as executor:
    future = executor.submit(scrape_test)
    test_rows = future.result()

# Create Polars DataFrame
if not test_rows:
    print("No data scraped.")
else:
    df_test = pl.DataFrame(test_rows).unique()
    print(f"\n[✓] Test complete! Scraped {len(df_test)} products.")

[*] Fetching Test Page: https://www.bigc.co.th/en/category/laundry?page=1&limit=90


[2026-08-24 03:19:27] INFO: Fetched (200) <GET https://www.bigc.co.th/en/category/laundry?page=1&limit=90> (referer: https://www.google.com/)
INFO:scrapling:Fetched (200) <GET https://www.bigc.co.th/en/category/laundry?page=1&limit=90> (referer: https://www.google.com/)


    -> Found 90 products
[*] Fetching Test Page: https://www.bigc.co.th/en/category/laundry?page=2&limit=90


[2026-08-24 03:19:48] INFO: Fetched (200) <GET https://www.bigc.co.th/en/category/laundry?page=2&limit=90> (referer: https://www.google.com/)
INFO:scrapling:Fetched (200) <GET https://www.bigc.co.th/en/category/laundry?page=2&limit=90> (referer: https://www.google.com/)


    -> Found 90 products

[✓] Test complete! Scraped 180 products.


In [66]:
df_test.to_pandas()

,product_name,promotion_price,original_price,condition
0,HYGIENE Color Bleach Laundry Stain Remover Flo...,65,72,Super Save
1,HYGIENE Expert Care Concentrated Fabric Soften...,47,60,Super Save
2,HYGIENE Expert Care Concentrated Fabric Soften...,47,60,Super Save
3,BIG C HAPPY PRICE Liquid Laundry Detergent Lov...,30,49,Super Save
4,DOWNY Garden Bloom Concentrated Fabric Softene...,187,210,Super Save
...,...,...,...,...
175,FINELINE Ironing Water Refill Violet Color 600...,24,None,None
176,HYGIENE Fabric Softener Soft white 1800 ml.,45,49,Super Save
177,COMFORT Pure Concentrated Fabric Softener 480 ...,115,120,Super Save
178,FRESH&SOFT Fabric Softener Lovely Kiss Scent 5...,10,14,Super Save


In [63]:
df_scrape = df_test.clone()

In [54]:
# @title udf data-prep
def re_evaluate_price(df: pl.DataFrame) -> pl.DataFrame:
    """
    Standardizes pricing logic:
    1. If original_price is missing, move the promotion_price to it.
    2. If promotion_price matches the original_price, set it to Null.
    """
    return (
        df.with_columns(
            # Step 1: Fix missing original prices by 'swapping' from promotion_price
            pl.when(pl.col("original_price").is_null() & pl.col("promotion_price").is_not_null())
            .then(pl.col("promotion_price"))
            .otherwise(pl.col("original_price"))
            .alias("original_price")
        )
        .with_columns(
            # Step 2: Now that original_price is populated, nullify redundant promotions
            pl.when(pl.col("promotion_price") == pl.col("original_price"))
            .then(None)
            .otherwise(pl.col("promotion_price"))
            .alias("promotion_price")
        )
    )


# @title udf Transform (Fixed Volume Extraction)
def parse_product_names(df: pl.DataFrame, shop_name: str) -> pl.DataFrame:
    """
    Pass any supermarket dataframe through this function to standardize the columns.
    Fixed to handle thousands separators (e.g., 1,300 ml).
    """
    # 1. Setup the dynamic date
    today_date = date.today().strftime("%Y-%m-%d")

    # 2. Updated patterns
    # Added [\d,.]+ to capture digits, commas, and dots
    # Updated pattern:
    # 1. Added (?)i for case-insensitivity
    # 2. Removed \b to ensure Thai characters don't get blocked by boundary logic
    quant_unit_pattern = r"(?i)([\d.]+)\s*(ML|G|KG|L|GRAMS?)"
    pack_pattern = r"(?i)(PACK\s*\d*\s*FREE\s*\d+|PACK\s*\d*\s*\+\s*\d+|PACK\s*\d+|TWINPACK|\bX\s*\d+\b|P?\d+\s*\+\s*\d+|\(\d+\+\d+\)|\d+\s*FREE\s*\d+|\bPACK\b)"

    # 3. Apply the Polars transformation
    return df.with_columns(
        pl.lit(today_date).alias("Date"),

        # Extract Brand
        pl.col("name").str.split(" ").list.first().alias("Brand"),

        # Fixed Volume Extraction:
        # 1. Extract the group (e.g., "1,300")
        # 2. Replace commas with nothing so it becomes "1300"
        # 3. Cast to Integer
        pl.col("name")
          .str.extract(quant_unit_pattern, 1)
          .str.replace_all(",", "")
          .cast(pl.Int64, strict=False)
          .alias("Volume"),

        # Extract Unit
        pl.col("name").str.extract(quant_unit_pattern, 2).str.to_uppercase().alias("Unit"),

        # Extract Pack size
        pl.col("name").str.extract(pack_pattern, 1).str.to_uppercase().alias("Pack"),

        # Add the dynamic Shop identifier
        pl.lit(shop_name).alias("Retailer")
    )

In [64]:
df_scrape = df_scrape.select([
    pl.col("product_name").alias("name"),
    # strict=False จะเปลี่ยนทุกอย่างที่แปลงเป็น Float ไม่ได้ให้เป็น Null
    pl.col("promotion_price").cast(pl.Float64, strict=False),
    pl.col("original_price").cast(pl.Float64, strict=False),
    pl.col("condition")
])
df_prep_big_c = re_evaluate_price(df_scrape)
df_trans_big_c = parse_product_names(df_prep_big_c, "BigC")

In [65]:
df_trans_big_c.head()

name,promotion_price,original_price,condition,Date,Brand,Volume,Unit,Pack,Retailer
str,f64,f64,str,str,str,i64,str,str,str
"""HYGIENE Color Bleach Laundry S…",65.0,72.0,"""Super Save""","""2026-08-24""","""HYGIENE""",1000,"""ML""",null,"""BigC"""
"""HYGIENE Expert Care Concentrat…",47.0,60.0,"""Super Save""","""2026-08-24""","""HYGIENE""",480,"""ML""",null,"""BigC"""
"""HYGIENE Expert Care Concentrat…",47.0,60.0,"""Super Save""","""2026-08-24""","""HYGIENE""",480,"""ML""",null,"""BigC"""
"""BIG C HAPPY PRICE Liquid Laund…",30.0,49.0,"""Super Save""","""2026-08-24""","""BIG""",700,"""ML""",null,"""BigC"""
"""DOWNY Garden Bloom Concentrate…",187.0,210.0,"""Super Save""","""2026-08-24""","""DOWNY""",2,"""L""",null,"""BigC"""


In [67]:
import time
import json
import re
import concurrent.futures
import polars as pl
from scrapling.fetchers import StealthyFetcher

# ----------------------------------------------------------------------
# 1. WATCHLIST URLS
# ----------------------------------------------------------------------
watchlist_urls = [
    # -- BIG C
    "https://www.bigc.co.th/en/product/fineline-liquid-laundry-detergent-sunny-gold-scent-550-ml.3791984",
    "https://www.bigc.co.th/en/product/fineline-plus-liquid-laundry-detergent-sunny-gold-scent-1250-ml.2155497",
    "https://www.bigc.co.th/en/product/hygiene-expert-wash-concentrate-liquid-detergent-milky-touch-600-ml.6394917",
    "https://www.bigc.co.th/en/product/hygiene-expert-wash-concentrate-liquid-detergent-milky-touch-1400-ml.6394919",
    "https://www.bigc.co.th/en/product/pao-win-wash-liquid-detergent-620-ml.12782",
    "https://www.bigc.co.th/en/product/pao-win-wash-concentrated-liquid-detergent-formula-1300-ml.34065",
    "https://www.bigc.co.th/en/product/pao-super-white-laundry-detergent-1800-g.5977",
    "https://www.bigc.co.th/en/product/pao-super-white-detergent-2400-g.3520",
    "https://www.bigc.co.th/en/product/attack-easy-detergent-happy-sweet-2500-g.47463",
    "https://www.bigc.co.th/en/product/hygiene-fabric-softener-expert-care-milky-touch-480-ml.12003",
    "https://www.bigc.co.th/en/product/hygiene-expert-care-concentrated-fabric-softener-milky-touch-scent-1000-ml-pack-2.1953035",
    "https://www.bigc.co.th/en/product/lipon-f-dishwashing-liquid-hygienic-formula-500-ml-pack-3.3639",
    "https://www.bigc.co.th/en/product/lipon-f-dishwashing-liquid-hygienic-formula-3200-ml.673",
    "https://www.bigc.co.th/en/product/pro-blue-plus-powder-laundry-detergent-standard-formula-2400-g.501",
    "https://www.bigc.co.th/en/product/lipon-f-sanitary-formula-dish-washing-liquid-refill-750-ml-pack-of-2.78902",
    "https://www.bigc.co.th/en/product/hygiene-fabric-softener-expert-care-tender-touch-480-ml-pack-2-free-1.32428",
    "https://www.bigc.co.th/en/product/attack-ez-conventional-detergent-happy-sweet-scent-1700-g.47863"
]

# ----------------------------------------------------------------------
# 2. UDF: EXTRACT PRODUCT FROM PAGE (JSON + HTML Fallback)
# ----------------------------------------------------------------------
def extract_watchlist_item(page_result, url: str) -> dict:
    """Extracts product data by first inspecting Next.js JSON, falling back to HTML."""
    name, promo_price, orig_price, condition = None, None, None, None

    # 1. Try extracting from Next.js embedded JSON
    try:
        raw_json = page_result.css('script#__NEXT_DATA__::text').get()
        if raw_json:
            json_data = json.loads(raw_json)
            page_props = json_data.get("props", {}).get("pageProps", {})
            product = page_props.get("product") or page_props.get("initialData", {}).get("product")

            if product:
                name = product.get("name") or product.get("title")
                promo_price = product.get("final_price") or product.get("special_price") or product.get("price")
                orig_price = product.get("price") or product.get("original_price")
                if promo_price == orig_price:
                    promo_price = None

                # Check for badges / conditions in JSON
                badges = product.get("badges") or product.get("promotions") or []
                if isinstance(badges, list):
                    condition = " | ".join([b.get("label", "") for b in badges if isinstance(b, dict) and b.get("label")])
    except Exception:
        pass

    # 2. Fallback to HTML selectors if JSON was missing or incomplete
    if not name:
        name_elem = page_result.css('h1::text').get() or page_result.css('title::text').get()
        name = name_elem.split(" - Big C")[0].strip() if name_elem else None

    if not promo_price:
        # Find prices formatted with ฿
        all_text = " ".join(page_result.css('body ::text').getall())
        price_matches = re.findall(r'฿\s*([\d,]+(?:\.\d+)?)', all_text)
        if price_matches:
            # First price on single product page is typically current price
            promo_price = price_matches[0].replace(",", "").strip()
            if len(price_matches) > 1 and float(price_matches[1].replace(",", "")) > float(promo_price):
                orig_price = price_matches[1].replace(",", "").strip()

    if not condition:
        badge_spans = page_result.css('main span::text').getall()
        valid_badges = [
            b.strip() for b in badge_spans
            if b.strip() and b.strip() != '฿'
            and not b.strip().startswith('-')
            and b.strip().lower() not in ["nextday", "next day", "pickup", "express", "add to cart", "share"]
        ]
        # Look for known promotional keywords
        promo_words = [b for b in valid_badges if any(k in b.lower() for k in ["save", "get", "free", "pack", "disc", "off", "deal"])]
        condition = " | ".join(list(dict.fromkeys(promo_words))) if promo_words else None

    return {
        "product_name": name.strip() if name else None,
        "promotion_price": str(promo_price) if promo_price is not None else None,
        "original_price": str(orig_price) if orig_price is not None else None,
        "condition": condition
    }

# ----------------------------------------------------------------------
# 3. UDF: SCRAPE WATCHLIST FUNCTION
# ----------------------------------------------------------------------
def scrape_watchlist(urls: list[str], delay: int = 1) -> list[dict]:
    """Iterates through a list of individual product URLs and scrapes their details."""
    scraped_data = []
    total = len(urls)

    for i, url in enumerate(urls, 1):
        print(f"[*] [{i}/{total}] Fetching: {url.split('/product/')[-1]}")
        try:
            page_result = StealthyFetcher.fetch(url, headless=True, network_idle=True)
            item_data = extract_watchlist_item(page_result, url)
            if item_data["product_name"]:
                scraped_data.append(item_data)
        except Exception as e:
            print(f"    [!] Error fetching {url}: {e}")

        time.sleep(delay)

    return scraped_data

# ----------------------------------------------------------------------
# 4. EXECUTION (ThreadPoolExecutor + Polars Pipeline)
# ----------------------------------------------------------------------
with concurrent.futures.ThreadPoolExecutor() as executor:
    future = executor.submit(scrape_watchlist, watchlist_urls)
    watchlist_rows = future.result()

if not watchlist_rows:
    print("No watchlist data scraped.")
else:
    df_watchlist = pl.DataFrame(watchlist_rows).unique()

    # Process through your existing cleaning pipeline
    df_watchlist_clean = df_watchlist.select([
        pl.col("product_name").alias("name"),
        pl.col("promotion_price").cast(pl.Float64, strict=False),
        pl.col("original_price").cast(pl.Float64, strict=False),
        pl.col("condition")
    ])

    df_prep_watchlist = re_evaluate_price(df_watchlist_clean)
    df_trans_watchlist = parse_product_names(df_prep_watchlist, "BigC")

    print(f"\n[✓] Watchlist complete! Scraped {len(df_trans_watchlist)} items.")
    df_trans_watchlist.to_pandas()

[*] [1/17] Fetching: fineline-liquid-laundry-detergent-sunny-gold-scent-550-ml.3791984


[2026-08-24 03:24:40] INFO: Fetched (200) <GET https://www.bigc.co.th/en/product/fineline-liquid-laundry-detergent-sunny-gold-scent-550-ml.3791984> (referer: https://www.google.com/)
INFO:scrapling:Fetched (200) <GET https://www.bigc.co.th/en/product/fineline-liquid-laundry-detergent-sunny-gold-scent-550-ml.3791984> (referer: https://www.google.com/)


[*] [2/17] Fetching: fineline-plus-liquid-laundry-detergent-sunny-gold-scent-1250-ml.2155497


[2026-08-24 03:24:52] INFO: Fetched (200) <GET https://www.bigc.co.th/en/product/fineline-plus-liquid-laundry-detergent-sunny-gold-scent-1250-ml.2155497> (referer: https://www.google.com/)
INFO:scrapling:Fetched (200) <GET https://www.bigc.co.th/en/product/fineline-plus-liquid-laundry-detergent-sunny-gold-scent-1250-ml.2155497> (referer: https://www.google.com/)


[*] [3/17] Fetching: hygiene-expert-wash-concentrate-liquid-detergent-milky-touch-600-ml.6394917


...

[2026-08-24 03:27:44] INFO: Fetched (200) <GET https://www.bigc.co.th/en/product/hygiene-fabric-softener-expert-care-tender-touch-480-ml-pack-2-free-1.32428> (referer: https://www.google.com/)
INFO:scrapling:Fetched (200) <GET https://www.bigc.co.th/en/product/hygiene-fabric-softener-expert-care-tender-touch-480-ml-pack-2-free-1.32428> (referer: https://www.google.com/)


[*] [17/17] Fetching: attack-ez-conventional-detergent-happy-sweet-scent-1700-g.47863


[2026-08-24 03:27:56] INFO: Fetched (200) <GET https://www.bigc.co.th/en/product/attack-ez-conventional-detergent-happy-sweet-scent-1700-g.47863> (referer: https://www.google.com/)
INFO:scrapling:Fetched (200) <GET https://www.bigc.co.th/en/product/attack-ez-conventional-detergent-happy-sweet-scent-1700-g.47863> (referer: https://www.google.com/)



[✓] Watchlist complete! Scraped 17 items.


TypeError: 'module' object is not callable

In [68]:
df_trans_watchlist

name,promotion_price,original_price,condition,Date,Brand,Volume,Unit,Pack,Retailer
str,f64,f64,str,str,str,i64,str,str,str
"""FINELINE Plus Liquid Laundry D…",null,179.0,"""Buy 1 Piece Get 1 Piece averag…","""2026-08-24""","""FINELINE""",1250,"""ML""",null,"""BigC"""
"""LIPON F Sanitary Formula Dish …",null,85.0,"""LIPON F Sanitary Formula Dish …","""2026-08-24""","""LIPON""",750,"""ML""","""PACK""","""BigC"""
"""HYGIENE Expert Care Concentrat…",189.0,219.0,"""HYGIENE Expert Care Concentrat…","""2026-08-24""","""HYGIENE""",1000,"""ML""","""PACK 2""","""BigC"""
"""HYGIENE Expert Wash Concentrat…",105.0,139.0,"""Super Save | *The company rese…","""2026-08-24""","""HYGIENE""",1400,"""ML""",null,"""BigC"""
"""HYGIENE Expert Wash Concentrat…",55.0,65.0,"""*The company reserves the righ…","""2026-08-24""","""HYGIENE""",600,"""ML""",null,"""BigC"""
…,…,…,…,…,…,…,…,…,…
"""HYGIENE Expert Care Concentrat…",47.0,60.0,"""*The company reserves the righ…","""2026-08-24""","""HYGIENE""",480,"""ML""",null,"""BigC"""
"""HYGIENE Expert Care Concentrat…",119.0,120.0,"""HYGIENE Expert Care Concentrat…","""2026-08-24""","""HYGIENE""",480,"""ML""","""PACK 2+1""","""BigC"""
"""ATTACK EZ Conventional Deterge…",89.0,115.0,"""*The company reserves the righ…","""2026-08-24""","""ATTACK""",1700,"""G""",null,"""BigC"""


In [69]:
list_to_search = [
# -- BIG C
'FINELINE Liquid Laundry Detergent Sunny Gold Scent 550 ml.',
'FINELINE Plus Liquid Laundry Detergent Sunny Gold Scent 1250 ml.',
'HYGIENE Expert Wash Concentrate Liquid Detergent Milky Touch 600 ml.',
'HYGIENE Expert Wash Concentrate Liquid Detergent Milky Touch 1400 ml.',
'PAO Win Wash Liquid Laundry Detergent 620 ml.',
'PAO Win Wash Liquid Laundry Detergent 1300 ml.',
'PAO Super White Laundry Detergent 1800 g.',
'PAO Super White Powder Laundry Detergent 2400 g.',
'ATTACK EASY DETERGENT HAPPY SWEET 2500 G',
'HYGIENE Expert Care Concentrated Fabric Softener Milky Touch Scent 480 ml.',
'HYGIENE Expert Care Concentrated Fabric Softener Milky Touch Scent 1000 ml. Pack 2',
'LIPON F Dishwashing Liquid Hygienic Formula 500 ml. Pack 3',
'LIPON F Dishwashing Liquid Hygienic Formula 3200 ml.',
'PRO Blue Plus Powder Laundry Detergent 2400 g.',
'LIPON F Sanitary Formula Dish Washing Liquid Refill 750 ml. Pack of 2',
'HYGIENE Expert Care Concentrated Fabric Softener Milky Touch Scent 480 ml. Pack 2+1',
'ATTACK Easy Conventional Detergent Happy Sweet Pink 1.7 kg x 1+1',
# Found this, above (Not Found)
'ATTACK EZ Conventional Detergent Happy Sweet Scent 1700 g.'
]

search_df = pl.DataFrame({"product_name": list_to_search})

# Join with df_trans_lotuss to get original_price and promotion_price
search_results_df = search_df.join(
    df_watchlist_clean.select(["name", "original_price", "promotion_price"]),
    left_on="product_name",
    right_on="name",
    how="left"
)
lotuss_names_set = set(df_watchlist_clean["name"].to_list())

search_results_df = search_results_df.with_columns(
    pl.col("product_name").is_in(lotuss_names_set).alias("Found")
).unique()

print("Search Results with Prices:")
print(search_results_df)

Search Results with Prices:
shape: (18, 4)
┌─────────────────────────────────┬────────────────┬─────────────────┬───────┐
│ product_name                    ┆ original_price ┆ promotion_price ┆ Found │
│ ---                             ┆ ---            ┆ ---             ┆ ---   │
│ str                             ┆ f64            ┆ f64             ┆ bool  │
╞═════════════════════════════════╪════════════════╪═════════════════╪═══════╡
│ PAO Win Wash Liquid Laundry De… ┆ null           ┆ 185.0           ┆ true  │
│ ATTACK EASY DETERGENT HAPPY SW… ┆ null           ┆ 159.0           ┆ true  │
│ HYGIENE Expert Care Concentrat… ┆ 60.0           ┆ 47.0            ┆ true  │
│ HYGIENE Expert Care Concentrat… ┆ 120.0          ┆ 119.0           ┆ true  │
│ HYGIENE Expert Care Concentrat… ┆ 219.0          ┆ 189.0           ┆ true  │
│ …                               ┆ …              ┆ …               ┆ …     │
│ HYGIENE Expert Wash Concentrat… ┆ 65.0           ┆ 55.0            ┆ true  │
│ LIPON F

In [70]:
search_results_df.to_pandas()

,product_name,original_price,promotion_price,Found
0,PAO Win Wash Liquid Laundry Detergent 1300 ml.,NaN,185.0,True
1,ATTACK EASY DETERGENT HAPPY SWEET 2500 G,NaN,159.0,True
2,HYGIENE Expert Care Concentrated Fabric Soften...,60.0,47.0,True
3,HYGIENE Expert Care Concentrated Fabric Soften...,120.0,119.0,True
4,HYGIENE Expert Care Concentrated Fabric Soften...,219.0,189.0,True
5,FINELINE Liquid Laundry Detergent Sunny Gold S...,89.0,49.0,True
6,ATTACK Easy Conventional Detergent Happy Sweet...,NaN,NaN,False
7,HYGIENE Expert Wash Concentrate Liquid Deterge...,139.0,105.0,True
8,PRO Blue Plus Powder Laundry Detergent 2400 g.,NaN,150.0,True
9,PAO Super White Powder Laundry Detergent 2400 g.,NaN,155.0,True


In [71]:
# @title Saving Excel
df_trans_big_c.write_excel(f"big_c_catalog_{today_date}.xlsx")
df_trans_watchlist.write_excel(f"big_c_watchlist_{today_date}.xlsx")
search_results_df.write_excel(f"search_result_big_c_{today_date}.xlsx")

In [72]:
# @title Automatic load compressed file to computer
import os
import zipfile
from google.colab import files
import datetime

# Get the current date for the filename
today_date = datetime.datetime.now().strftime("%Y-%m-%d")

# Get all files in the current directory, excluding common Colab configuration files
files_to_zip = [f for f in os.listdir('.') if os.path.isfile(f) and not f.startswith('.') and not f.startswith('sample_data')]

print("Iterating through files in the current directory:")
for file in files_to_zip:
    print(f"- {file}")

zip_filename = f'bigc_{today_date}.zip'

with zipfile.ZipFile(zip_filename, 'w') as zipf:
    for file in files_to_zip:
        zipf.write(file)

print(f"\n'{zip_filename}' created successfully. Downloading now...")
files.download(zip_filename)

Iterating through files in the current directory:
- search_result_big_c_2026-08-24.xlsx
- big_c_watchlist_2026-08-24.xlsx
- bigc_products.csv
- big_c_catalog_2026-08-24.xlsx
- bigc_products.xlsx

'bigc_2026-08-24.zip' created successfully. Downloading now...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>